
# Likelihood Analysis — **Four-Lineage Focus (No GLOBAL)**

This notebook reads the **likelihood stage** outputs under `results/likelihood/tables` and builds an exhaustive analysis focused on **exactly four lineages** that you specify (e.g., `B.1.1.7`, `B.1.351`, `P.1`, `B.1.617.2`).

**Key behaviors**

- **No `GLOBAL` lineage** in any plot or table (dropped by default).
- Operates on **exactly four** target lineages. If any of those are missing in your results, it will **auto-select the Top‑4** lineages globally and inform you.
- All **other** lineages (besides the four) can be **aggregated into `Other`** (optional) and θ are **renormalized per sample** over the displayed set.
- Saves figures and summary CSVs if you set `SAVE_FIGS=True` below.
- Uses **matplotlib only** (no seaborn).

> You can rerun this notebook at any time after re-running your likelihood stage.


In [ ]:

# === Configuration ===
import os
import re
from pathlib import Path

# Root where all likelihood outputs live
LIKELIHOOD_DIR = r"C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\likelihood"

# Subfolder where tables are stored (CSV + TEX files)
ALT_TABLES_DIR = os.path.join(LIKELIHOOD_DIR, "tables")

# Where this notebook will export figures and summary tables
OUTPUT_DIR = r"C:\Users\osoom\OneDrive\Desktop\ssssssss\oxbio-variant-forecasting\results\likelihood_analysis_nb"

# Optional: regex to subset sites (e.g., r"^NYC|^SF"); set to None for all
SITE_FILTER_REGEX = None

# Plot and computation controls
TOP_LINEAGES = 8              # used only for fallbacks / context summaries
MAX_SITES_PLOTS = 16          # per-site plots cap
DOWNSAMPLE_SCATTER = 150_000  # max points for hexbin diagnostics
SAVE_FIGS = False             # export PNGs + CSVs under OUTPUT_DIR
RANDOM_SEED = 12345           # reproducibility for downsampling

# ---- Four-lineage focus ----
# Put YOUR exact four here. If any are missing, the notebook will auto-pick top-4.
EXACT_LINEAGES = ["B.1.1.7", "B.1.351", "P.1", "B.1.617.2"]

# Aggregate all non-target lineages into "Other" (True/False)
AGGREGATE_OTHERS = True

# Always drop any rows labeled lineage == "GLOBAL" from displays
DROP_GLOBAL = True

# Rolling window for growth rate (in days) when computing d/dt logit(share)
GROWTH_WINDOW_DAYS = 7

# Minimum number of distinct dates in a site to include it in time-series plots
MIN_DATES_PER_SITE = 5

# If True, reweight site/day averages by 'median_coverage' when available
WEIGHT_BY_MEDIAN_COV = True

# End of config
print("Config loaded.")


In [ ]:

import math
import warnings
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(RANDOM_SEED)

def _ensure_dir(p: str):
    Path(p).mkdir(parents=True, exist_ok=True)

def _savefig(fig, name: str, dpi=140):
    if not SAVE_FIGS:
        return
    _ensure_dir(OUTPUT_DIR)
    out_png = os.path.join(OUTPUT_DIR, f"{name}.png")
    fig.savefig(out_png, dpi=dpi, bbox_inches="tight")
    print(f"[saved] {out_png}")

def _write_csv(df: pd.DataFrame, name: str):
    if not SAVE_FIGS:
        return
    _ensure_dir(OUTPUT_DIR)
    out_csv = os.path.join(OUTPUT_DIR, f"{name}.csv")
    df.to_csv(out_csv, index=False)
    print(f"[saved] {out_csv}")

def _read_csv_safe(p: str) -> pd.DataFrame:
    try:
        return pd.read_csv(p)
    except Exception as e:
        raise FileNotFoundError(f"Failed to read: {p}\n{e}")

def _parse_date_col(df: pd.DataFrame, col: str = "date"):
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")
        df = df.dropna(subset=[col]).reset_index(drop=True)
    return df

def _maybe_subset_sites(df: pd.DataFrame) -> pd.DataFrame:
    if SITE_FILTER_REGEX is None:
        return df
    pat = re.compile(SITE_FILTER_REGEX)
    return df[df["site_id"].astype(str).str.match(pat)].copy()

print("Utilities ready.")


In [ ]:

# ---- Load core tables from results/likelihood/tables ----
tables_dir = Path(ALT_TABLES_DIR)
if not tables_dir.exists():
    raise FileNotFoundError(f"Tables folder not found: {tables_dir}")

print("Reading tables from:", tables_dir)

theta_path = tables_dir / "theta_estimates.csv"
resid_path = tables_dir / "residuals.csv"

theta_df = _read_csv_safe(str(theta_path))
theta_df = _parse_date_col(theta_df, "date")
theta_df["site_id"] = theta_df["site_id"].astype(str)
theta_df["sample_id"] = theta_df["sample_id"].astype(str)
theta_df["lineage"] = theta_df["lineage"].astype(str)
theta_df["theta"] = pd.to_numeric(theta_df["theta"], errors="coerce").fillna(0.0).clip(0.0, 1.0)

# Optional tables
zdiag_df = None
if (tables_dir / "zscore_diagnostics.csv").exists():
    zdiag_df = _read_csv_safe(str(tables_dir / "zscore_diagnostics.csv"))
    zdiag_df = _parse_date_col(zdiag_df, "date")

theta_sd_df = None
if (tables_dir / "theta_uncertainty.csv").exists():
    theta_sd_df = _read_csv_safe(str(tables_dir / "theta_uncertainty.csv"))
    theta_sd_df = _parse_date_col(theta_sd_df, "date")

obj_df = None
if (tables_dir / "objective_trace.csv").exists():
    obj_df = _read_csv_safe(str(tables_dir / "objective_trace.csv"))

ov_df = None
if (tables_dir / "overlap_matrix.csv").exists():
    ov_df = _read_csv_safe(str(tables_dir / "overlap_matrix.csv"))

residuals_df = None
if resid_path.exists():
    residuals_df = _read_csv_safe(str(resid_path))
    residuals_df = _parse_date_col(residuals_df, "date")

print("Loaded:")
print(" - theta_estimates:", theta_df.shape)
print(" - residuals:", None if residuals_df is None else residuals_df.shape)
print(" - zscore_diagnostics:", None if zdiag_df is None else zdiag_df.shape)
print(" - theta_uncertainty:", None if theta_sd_df is None else theta_sd_df.shape)
print(" - objective_trace:", None if obj_df is None else obj_df.shape)
print(" - overlap_matrix:", None if ov_df is None else ov_df.shape)

# Optional med coverage present?
has_med_cov = "median_coverage" in theta_df.columns
print("median_coverage present in theta_estimates:", has_med_cov)


In [ ]:

# ---- Enforce four-lineage focus ----
df0 = theta_df.copy()
if DROP_GLOBAL:
    df0 = df0[df0["lineage"].str.upper() != "GLOBAL"].copy()

# If any of the EXACT_LINEAGES are missing, fallback to top-4 by global weight
lineages_present = sorted(df0["lineage"].unique())
missing = [L for L in EXACT_LINEAGES if L not in lineages_present]

if missing:
    print(f"WARNING: Missing target lineages: {missing}")
    # Pick top-4 globally by coverage-weighted average share
    # Use weights from theta_estimates.median_coverage if available, otherwise equal weight per sample
    tmp = df0.copy()
    if WEIGHT_BY_MEDIAN_COV and "median_coverage" in tmp.columns:
        w = tmp["median_coverage"].fillna(1.0).clip(lower=1.0)
    else:
        # fall back to 1.0 per sample
        w = pd.Series(1.0, index=tmp.index)
    g = tmp.groupby("lineage").apply(lambda g: np.average(g["theta"].to_numpy(float), weights=w.loc[g.index].to_numpy(float))).reset_index(name="w_mean_theta")
    g = g.sort_values("w_mean_theta", ascending=False)
    EXACT4 = g["lineage"].head(4).tolist()
    print(f"Auto-selected Top-4 lineages: {EXACT4}")
else:
    EXACT4 = list(EXACT_LINEAGES)

print("Using these FOUR lineages:", EXACT4)


In [ ]:

# ---- Build display set: four lineages (+ optional Other) ----
df = theta_df.copy()
if DROP_GLOBAL:
    df = df[df["lineage"].str.upper() != "GLOBAL"].copy()

df = _maybe_subset_sites(df)
if df.empty:
    raise RuntimeError("No rows left after site filter / GLOBAL drop.")

# Ensure theta is numeric and clipped
df["theta"] = pd.to_numeric(df["theta"], errors="coerce").fillna(0.0).clip(0.0, 1.0)

# Keep only four + aggregate others
target_set = set(EXACT4)
df["lineage"] = df["lineage"].astype(str)

if AGGREGATE_OTHERS:
    # Sum theta of non-target lineages into 'Other'
    rest = df[~df["lineage"].isin(target_set)].copy()
    if not rest.empty:
        rest_sum = (rest.groupby(["site_id","date","sample_id"], as_index=False)
                        .agg(theta=("theta","sum")))
        rest_sum["lineage"] = "Other"
        keep = df[df["lineage"].isin(target_set)].copy()
        df_disp = pd.concat([keep, rest_sum], ignore_index=True)
    else:
        df_disp = df[df["lineage"].isin(target_set)].copy()
else:
    df_disp = df[df["lineage"].isin(target_set)].copy()

# Renormalize per (site, date, sample_id)
sums = df_disp.groupby(["site_id","date","sample_id"], as_index=False)["theta"].sum().rename(columns={"theta":"theta_total"})
df_disp = df_disp.merge(sums, on=["site_id","date","sample_id"], how="left")
df_disp["theta"] = np.divide(df_disp["theta"], np.maximum(df_disp["theta_total"], 1e-12))
df_disp = df_disp.drop(columns=["theta_total"])

# Sort columns
df_disp = df_disp[["site_id","date","sample_id","lineage","theta"] + ([c for c in df.columns if c not in {"site_id","date","sample_id","lineage","theta"}])]

print("Display set (first 10):")
display(df_disp.head(10))


In [ ]:

# ---- Weights for averaging samples within site-day ----
if WEIGHT_BY_MEDIAN_COV and "median_coverage" in theta_df.columns:
    site_day_w = (theta_df.groupby(["site_id","date"], as_index=False)["median_coverage"]
                         .median()
                         .rename(columns={"median_coverage":"site_day_weight"}))
else:
    # fall back to equal weights
    tmp = theta_df[["site_id","date"]].drop_duplicates().copy()
    tmp["site_day_weight"] = 1.0
    site_day_w = tmp

# Compute site/day lineage shares (weighted mean across samples for same site-day)
def _site_day_lineage(df_in: pd.DataFrame) -> pd.DataFrame:
    merge = df_in.merge(site_day_w, on=["site_id","date"], how="left")
    w = merge["site_day_weight"].fillna(1.0).to_numpy(float)
    out = (merge.groupby(["site_id","date","lineage"])
                 .apply(lambda g: np.average(g["theta"].to_numpy(float), weights=g["site_day_weight"].fillna(1.0).to_numpy(float)))
                 .reset_index(name="share"))
    return out

site_day = _site_day_lineage(df_disp)

# Pivot to wide for convenience (columns: the four + maybe Other)
lineage_cols = sorted(site_day["lineage"].unique(), key=lambda x: (x!="Other", x))
wide_site_day = site_day.pivot_table(index=["site_id","date"], columns="lineage", values="share", fill_value=0.0).reset_index()
wide_site_day.columns.name = None

# Only keep sites with enough dates
site_counts = wide_site_day.groupby("site_id")["date"].nunique().reset_index(name="n_dates")
keep_sites = site_counts[site_counts["n_dates"] >= MIN_DATES_PER_SITE]["site_id"].tolist()
wide_site_day = wide_site_day[wide_site_day["site_id"].isin(keep_sites)].copy()
site_day = site_day[site_day["site_id"].isin(keep_sites)].copy()

print(f"Sites kept (>= {MIN_DATES_PER_SITE} dates): {len(keep_sites)}")

# Global aggregate per date (weighted by site-day weights)
merge_g = wide_site_day.merge(site_day_w, on=["site_id","date"], how="left")
merge_g["site_day_weight"] = merge_g["site_day_weight"].fillna(1.0)

def _weighted_mean_per_date(df_in: pd.DataFrame, cols: list) -> pd.DataFrame:
    out_rows = []
    for dt, g in df_in.groupby("date"):
        w = g["site_day_weight"].to_numpy(float)
        row = {"date": dt}
        for c in cols:
            if c in g.columns:
                row[c] = float(np.average(g[c].to_numpy(float), weights=w))
        out_rows.append(row)
    return pd.DataFrame(out_rows).sort_values("date")

global_ts = _weighted_mean_per_date(merge_g, [c for c in lineage_cols if c != "site_id" and c != "date"])

print("Global time series (first rows):")
display(global_ts.head())

# Last-date composition (global)
if not global_ts.empty:
    last_row = global_ts.iloc[[-1]].copy()
    last_row.insert(0, "label", "last_date")
    _write_csv(last_row, "global_last_date_composition")


In [ ]:

# Save wide tables if requested
_write_csv(wide_site_day.sort_values(["site_id","date"]), "site_day_lineage_shares")
_write_csv(global_ts, "global_lineage_shares")

# Also export per-site last-date composition
last_date_per_site = (wide_site_day.sort_values("date").groupby("site_id").tail(1).reset_index(drop=True))
_write_csv(last_date_per_site, "per_site_last_date_composition")

print("Core summary tables prepared.")


In [ ]:

def plot_global_stacked(global_df: pd.DataFrame, cols: list, title: str = "Global lineage shares"):
    if global_df.empty:
        print("No data for global stacked plot.")
        return None
    dfp = global_df.copy()
    dfp = dfp.sort_values("date")
    y = [dfp[c].to_numpy(float) if c in dfp.columns else np.zeros(len(dfp)) for c in cols]
    fig = plt.figure(figsize=(11, 5.5))
    plt.stackplot(dfp["date"].to_numpy(), *y, labels=cols, step="pre")
    plt.title(title); plt.xlabel("date"); plt.ylabel("share"); plt.ylim(0, 1.0)
    plt.legend(loc="upper left", ncol=min(5, len(cols)))
    fig.tight_layout()
    return fig

def plot_last_date_bar(global_df: pd.DataFrame, cols: list, title: str = "Global composition (last date)"):
    if global_df.empty:
        print("No data for last-date composition.")
        return None
    last = global_df.sort_values("date").iloc[-1]
    vals = [float(last.get(c, 0.0)) for c in cols]
    xs = np.arange(len(cols))
    fig = plt.figure(figsize=(8, 4.5))
    plt.bar(xs, vals)
    plt.xticks(xs, cols, rotation=30, ha="right")
    plt.ylabel("share"); plt.title(title); plt.ylim(0, 1.0)
    fig.tight_layout()
    return fig

def plot_site_stacked(wide_df: pd.DataFrame, cols: list, site: str):
    dfx = wide_df[wide_df["site_id"] == site].sort_values("date").copy()
    if dfx.empty:
        return None
    y = [dfx[c].to_numpy(float) if c in dfx.columns else np.zeros(len(dfx)) for c in cols]
    fig = plt.figure(figsize=(11, 4.6))
    plt.stackplot(dfx["date"].to_numpy(), *y, labels=cols, step="pre")
    plt.title(f"{site} — lineage shares"); plt.xlabel("date"); plt.ylabel("share"); plt.ylim(0, 1.0)
    plt.legend(loc="upper left", ncol=min(5, len(cols)))
    fig.tight_layout()
    return fig

def plot_lineage_cross_sites(wide_df: pd.DataFrame, lineage: str):
    fig = plt.figure(figsize=(11, 4.6))
    for sid, g in wide_df.groupby("site_id"):
        g = g.sort_values("date")
        if lineage in g.columns:
            plt.plot(g["date"], g[lineage], alpha=0.7)
    plt.title(f"Cross-site trajectories — {lineage}")
    plt.xlabel("date"); plt.ylabel("share"); plt.ylim(0, 1.0)
    fig.tight_layout()
    return fig

def _downsample_idx(n: int, k: int, rng=np.random.default_rng(123)):
    if n <= k:
        return np.arange(n)
    return rng.choice(np.arange(n), size=k, replace=False)

def plot_hexbin_pred_vs_obs(residuals_df: pd.DataFrame, max_pts: int = 150_000):
    if residuals_df is None or residuals_df.empty:
        print("Residuals not available.")
        return None
    df = residuals_df.dropna(subset=["pred_af","obs_af"]).copy()
    idx = _downsample_idx(len(df), max_pts, rng)
    d = df.iloc[idx]
    fig = plt.figure(figsize=(6.8, 5.6))
    plt.hexbin(d["pred_af"], d["obs_af"], gridsize=55, extent=(0,1,0,1), mincnt=2)
    plt.plot([0,1],[0,1], "--", lw=1.0)
    plt.xlabel("predicted AF"); plt.ylabel("observed AF"); plt.title("Pred vs Obs AF (hexbin)")
    fig.tight_layout()
    return fig

def plot_hexbin_resid_vs_cov(residuals_df: pd.DataFrame, max_pts: int = 150_000):
    if residuals_df is None or residuals_df.empty:
        print("Residuals not available.")
        return None
    df = residuals_df.dropna(subset=["pred_af","obs_af","coverage"]).copy()
    df["resid"] = df["obs_af"] - df["pred_af"]
    idx = _downsample_idx(len(df), max_pts, rng)
    d = df.iloc[idx]
    fig = plt.figure(figsize=(6.8, 5.6))
    plt.hexbin(np.maximum(d["coverage"].to_numpy(float), 1.0), d["resid"].to_numpy(float), gridsize=55, xscale="log", mincnt=2)
    plt.axhline(0, lw=1.0)
    plt.xlabel("coverage (log)"); plt.ylabel("obs - pred"); plt.title("Residual vs coverage (hexbin)")
    fig.tight_layout()
    return fig

def plot_calibration_curve(residuals_df: pd.DataFrame, bins: int = 25):
    if residuals_df is None or residuals_df.empty:
        print("Residuals not available.")
        return None
    df = residuals_df.dropna(subset=["pred_af","obs_af"]).copy()
    df["pred_bin"] = pd.qcut(df["pred_af"], q=bins, duplicates="drop")
    agg = df.groupby("pred_bin", observed=True).agg(pred=("pred_af","mean"), obs=("obs_af","mean")).reset_index(drop=True)
    fig = plt.figure(figsize=(6.8, 5.6))
    plt.plot([0,1],[0,1], "--", lw=1.0, label="y=x")
    plt.plot(agg["pred"], agg["obs"], marker="o", lw=1.2)
    plt.xlabel("Pred AF (bin mean)"); plt.ylabel("Observed AF (mean)"); plt.title("Calibration")
    plt.legend()
    fig.tight_layout()
    return fig

def plot_hist(z, title="Z-score histogram"):
    fig = plt.figure(figsize=(6.8, 5.0))
    plt.hist(z, bins=min(70, max(30, int(np.sqrt(len(z))))) if len(z) else 30, density=True, alpha=0.8)
    x = np.linspace(-4,4,401)
    plt.plot(x, (1/np.sqrt(2*np.pi))*np.exp(-0.5*x**2), lw=1.2)
    plt.title(title); plt.xlabel("z"); plt.ylabel("density")
    fig.tight_layout()
    return fig

def plot_qq(z, title="Z-score QQ"):
    if len(z) == 0:
        print("No z for QQ.")
        return None
    z = np.sort(z)
    ppos = (np.arange(1, len(z) + 1) - 0.5) / len(z)
    p = np.clip(ppos, 1e-12, 1-1e-12)
    q = np.log(p/(1-p)) / 1.702
    lim = 1.05 * max(1.0, np.nanmax(np.abs(np.concatenate([q, z]))))
    fig = plt.figure(figsize=(6.8, 5.0))
    plt.plot(q, z, marker="o", linestyle="none", ms=2.5, alpha=0.6)
    plt.plot([-lim, lim], [-lim, lim], lw=1.0)
    plt.title(title); plt.xlabel("theoretical"); plt.ylabel("empirical"); plt.xlim(-lim, lim); plt.ylim(-lim, lim)
    fig.tight_layout()
    return fig

print("Plotting helpers ready.")


In [ ]:

# ---- Global stacked & last-date composition ----
cols_plot = [c for c in ["Other"] + EXACT4 if c in global_ts.columns] if AGGREGATE_OTHERS else [c for c in EXACT4 if c in global_ts.columns]

fig = plot_global_stacked(global_ts, cols_plot, "Global lineage shares (four-lineage focus)")
if fig:
    _savefig(fig, "global_stacked_shares")
    plt.show()
    plt.close(fig)

fig = plot_last_date_bar(global_ts, cols_plot, "Global composition (last date)")
if fig:
    _savefig(fig, "global_last_date_composition_bar")
    plt.show()
    plt.close(fig)


In [ ]:

# ---- Per-site stacked areas (one figure per site, up to MAX_SITES_PLOTS) ----
sites_to_plot = sorted(wide_site_day["site_id"].unique())[:MAX_SITES_PLOTS]
for sid in sites_to_plot:
    fig = plot_site_stacked(wide_site_day, cols_plot, sid)
    if fig:
        _savefig(fig, f"site_stacked__{sid}")
        plt.show()
        plt.close(fig)

# ---- Cross-site trajectories per lineage (one fig per lineage) ----
for lin in [c for c in cols_plot if c != "Other"]:
    fig = plot_lineage_cross_sites(wide_site_day, lin)
    if fig:
        _savefig(fig, f"cross_site__{lin}")
        plt.show()
        plt.close(fig)


In [ ]:

# ---- Residual diagnostics ----
if residuals_df is not None and not residuals_df.empty:
    fig = plot_hexbin_pred_vs_obs(residuals_df, max_pts=DOWNSAMPLE_SCATTER)
    if fig:
        _savefig(fig, "diag_hexbin_pred_vs_obs")
        plt.show(); plt.close(fig)

    fig = plot_hexbin_resid_vs_cov(residuals_df, max_pts=DOWNSAMPLE_SCATTER)
    if fig:
        _savefig(fig, "diag_hexbin_resid_vs_cov")
        plt.show(); plt.close(fig)

    fig = plot_calibration_curve(residuals_df, bins=25)
    if fig:
        _savefig(fig, "diag_calibration_curve")
        plt.show(); plt.close(fig)

# Z-scores if available
if zdiag_df is not None and not zdiag_df.empty and "z" in zdiag_df.columns:
    z = zdiag_df["z"].to_numpy(float)
    z = z[np.isfinite(z)]
    fig = plot_hist(z, "Z-score histogram")
    if fig:
        _savefig(fig, "diag_z_hist")
        plt.show(); plt.close(fig)
    fig = plot_qq(z, "Z-score QQ")
    if fig:
        _savefig(fig, "diag_z_qq")
        plt.show(); plt.close(fig)


In [ ]:

# ---- Growth & peak timing utilities ----
def _logit(x):
    x = np.clip(x, 1e-9, 1-1e-9)
    return np.log(x/(1-x))

def _rolling_slope(dates, series, window_days=7):
    # Convert to ordinal days, fit slope over rolling window by OLS
    if len(series) < 3:
        return np.array([np.nan]*len(series))
    days = pd.to_datetime(dates).map(pd.Timestamp.toordinal).to_numpy(float)
    y = series.astype(float)
    res = np.full(len(series), np.nan, float)
    for i in range(len(series)):
        # window centered at i
        t0 = days[i] - window_days/2.0
        t1 = days[i] + window_days/2.0
        m = (days >= t0) & (days <= t1) & np.isfinite(y)
        if np.sum(m) >= 3:
            X = np.vstack([np.ones(np.sum(m)), days[m]]).T
            beta, *_ = np.linalg.lstsq(X, y[m], rcond=None)
            res[i] = beta[1]  # slope per day
    return res

def compute_growth_and_peaks(wide_df: pd.DataFrame, cols: list, window_days: int = 7):
    rows_growth = []
    rows_peak = []
    for sid, g in wide_df.groupby("site_id"):
        g = g.sort_values("date").reset_index(drop=True)
        for lin in cols:
            if lin not in g.columns:
                continue
            s = g[lin].to_numpy(float)
            if len(s) < 3:
                continue
            lo = _logit(s)
            slope = _rolling_slope(g["date"], lo, window_days=window_days)
            for dt, sl in zip(g["date"], slope):
                rows_growth.append({"site_id": sid, "date": dt, "lineage": lin, "logit_slope_per_day": float(sl)})
            # Peak timing
            pk_idx = int(np.nanargmax(s))
            rows_peak.append({"site_id": sid, "lineage": lin, "peak_date": g.loc[pk_idx, "date"], "peak_share": float(s[pk_idx])})
    return pd.DataFrame(rows_growth), pd.DataFrame(rows_peak)

growth_df, peak_df = compute_growth_and_peaks(wide_site_day, [c for c in cols_plot if c != "Other"], window_days=GROWTH_WINDOW_DAYS)

_write_csv(growth_df, "growth_logit_slope")
_write_csv(peak_df, "peak_timing")

# ---- Plot: distribution of logit slopes per lineage ----
for lin in sorted(growth_df["lineage"].unique()) if not growth_df.empty else []:
    g = growth_df[growth_df["lineage"] == lin].copy()
    vals = g["logit_slope_per_day"].replace([np.inf,-np.inf], np.nan).dropna().to_numpy(float)
    if vals.size == 0:
        continue
    fig = plt.figure(figsize=(7.2, 5.0))
    plt.hist(vals, bins=60, density=True, alpha=0.85)
    plt.title(f"Growth (d/dt logit share) — {lin}")
    plt.xlabel("slope per day"); plt.ylabel("density")
    fig.tight_layout()
    _savefig(fig, f"growth_slope_hist__{lin}")
    plt.show(); plt.close(fig)

# Effective doubling time of odds = ln(2)/slope
for lin in sorted(growth_df["lineage"].unique()) if not growth_df.empty else []:
    g = growth_df[growth_df["lineage"] == lin].copy()
    s = g["logit_slope_per_day"].replace([np.inf,-np.inf], np.nan).to_numpy(float)
    s = s[np.isfinite(s) & (s > 0)]
    if s.size == 0: 
        continue
    dt = np.log(2.0) / s
    fig = plt.figure(figsize=(7.2, 5.0))
    plt.hist(np.clip(dt, 0, 120), bins=60, density=True, alpha=0.85)
    plt.title(f"Doubling time of odds (days) — {lin}")
    plt.xlabel("days"); plt.ylabel("density")
    fig.tight_layout()
    _savefig(fig, f"doubling_time_odds_hist__{lin}")
    plt.show(); plt.close(fig)

# Peak timing histogram per lineage
if not peak_df.empty:
    for lin in sorted(peak_df["lineage"].unique()):
        g = peak_df[peak_df["lineage"] == lin].copy()
        # Convert to ordinal month for a clean histogram
        months = g["peak_date"].dt.to_period("M").astype(str).to_numpy()
        # Count per month
        counts = pd.Series(months).value_counts().sort_index()
        fig = plt.figure(figsize=(8.2, 4.6))
        xs = np.arange(len(counts))
        plt.bar(xs, counts.values)
        plt.xticks(xs, counts.index, rotation=45, ha="right")
        plt.ylabel("site count"); plt.title(f"Peak timing by month — {lin}")
        fig.tight_layout()
        _savefig(fig, f"peak_timing_by_month__{lin}")
        plt.show(); plt.close(fig)


In [ ]:

# ---- Cross-lineage correlation of daily Δshare within each site ----
def per_site_delta_cor(wide_df: pd.DataFrame, cols: list):
    mats = []
    for sid, g in wide_df.groupby("site_id"):
        g = g.sort_values("date").reset_index(drop=True)
        if len(g) < 3:
            continue
        X = []
        for c in cols:
            if c in g.columns:
                x = g[c].to_numpy(float)
                dx = np.diff(x)
                X.append(dx)
        if len(X) < 2:
            continue
        X = np.vstack(X)
        C = np.corrcoef(X)
        mats.append(C)
    if not mats:
        return None
    return np.nanmean(np.stack(mats, axis=0), axis=0)

cols_corr = [c for c in cols_plot if c != "Other"]
Cavg = per_site_delta_cor(wide_site_day, cols_corr)
if Cavg is not None:
    fig = plt.figure(figsize=(6.4, 5.6))
    im = plt.imshow(np.clip(Cavg, -1, 1), vmin=-1, vmax=1, aspect="equal")
    plt.xticks(np.arange(len(cols_corr)), cols_corr, rotation=30, ha="right")
    plt.yticks(np.arange(len(cols_corr)), cols_corr)
    plt.colorbar(im, fraction=0.046, pad=0.04, label="corr(Δshare, Δshare)")
    plt.title("Average cross-lineage correlation (Δshare) across sites")
    fig.tight_layout()
    _savefig(fig, "cross_lineage_delta_corr_avg")
    plt.show(); plt.close(fig)

    # Save matrix
    dfC = pd.DataFrame(Cavg, index=cols_corr, columns=cols_corr)
    _write_csv(dfC.reset_index().rename(columns={"index":"lineage"}), "cross_lineage_delta_corr_matrix")
else:
    print("Not enough data to compute cross-lineage correlations.")


In [ ]:

# ---- Approximate uncertainty bands (if theta_uncertainty.csv present) ----
if theta_sd_df is not None and not theta_sd_df.empty:
    # Pick site with most days
    counts = wide_site_day.groupby("site_id")["date"].nunique().reset_index(name="n")
    if not counts.empty:
        site_star = counts.sort_values("n", ascending=False)["site_id"].iloc[0]
        print("Uncertainty bands for site:", site_star)

        # Build per-date mean theta and SD for each lineage (approx by averaging SDs)
        dtheta = df_disp[df_disp["site_id"] == site_star].copy()
        sd = theta_sd_df.copy()
        if DROP_GLOBAL:
            sd = sd[sd["lineage"].str.upper() != "GLOBAL"].copy()
        sd = sd[sd["lineage"].isin(set(cols_corr) | {"Other"})] if AGGREGATE_OTHERS else sd[sd["lineage"].isin(cols_corr)]
        # Approximate: average SD per site-date-lineage
        sdm = sd.groupby(["site_id","date","lineage"])["theta_sd"].mean().reset_index()

        m = dtheta.groupby(["site_id","date","lineage"])["theta"].mean().reset_index()
        m = m.merge(sdm, on=["site_id","date","lineage"], how="left")
        m = m[m["lineage"].isin([c for c in cols_plot if c != "Other"])].copy()

        for lin in sorted(m["lineage"].unique()):
            g = m[m["lineage"] == lin].sort_values("date").copy()
            if g.empty:
                continue
            mu = g["theta"].to_numpy(float)
            sdv = g["theta_sd"].fillna(np.nanmedian(g["theta_sd"])).to_numpy(float)
            lo = np.clip(mu - 1.96*sdv, 0, 1)
            hi = np.clip(mu + 1.96*sdv, 0, 1)
            fig = plt.figure(figsize=(10.8, 4.6))
            plt.plot(g["date"], mu, lw=1.5, label=lin)
            plt.fill_between(g["date"], lo, hi, alpha=0.25, step="mid")
            plt.title(f"{site_star} — {lin} (mean ± 1.96·sd approx)")
            plt.xlabel("date"); plt.ylabel("share"); plt.ylim(0,1)
            plt.legend()
            fig.tight_layout()
            _savefig(fig, f"uncertainty_band__{site_star}__{lin}")
            plt.show(); plt.close(fig)
else:
    print("theta_uncertainty.csv not available; skipping uncertainty bands.")


In [ ]:

# ---- Objective traces (if available) ----
if obj_df is not None and not obj_df.empty and {"site_id","iter","objective"} <= set(obj_df.columns):
    # Per-site plot (up to MAX_SITES_PLOTS)
    sites_obj = sorted(obj_df["site_id"].astype(str).unique())[:MAX_SITES_PLOTS]
    for sid in sites_obj:
        g = obj_df[obj_df["site_id"].astype(str) == sid].copy()
        if g.empty:
            continue
        fig = plt.figure(figsize=(7.5, 4.6))
        plt.plot(g["iter"].to_numpy(int), g["objective"].to_numpy(float))
        plt.xlabel("iteration"); plt.ylabel("objective"); plt.title(f"Objective trace — {sid}")
        fig.tight_layout()
        _savefig(fig, f"objective_trace__{sid}")
        plt.show(); plt.close(fig)

    # Median across sites per iteration
    gmed = (obj_df.groupby("iter", as_index=False)["objective"].median())
    fig = plt.figure(figsize=(7.5, 4.6))
    plt.plot(gmed["iter"], gmed["objective"])
    plt.xlabel("iteration"); plt.ylabel("median objective"); plt.title("Objective (median across sites)")
    fig.tight_layout()
    _savefig(fig, "objective_trace_median")
    plt.show(); plt.close(fig)
else:
    print("No objective_trace found.")


In [ ]:

# ---- Signature overlap (if available), restricted to four lineages ----
if ov_df is not None and not ov_df.empty:
    # Expect a square matrix with lineages as both index and columns
    # If file saved as a full matrix with headers, try direct parse:
    try:
        ov_df = ov_df.set_index(ov_df.columns[0])
    except Exception:
        pass
    # Keep the intersection of EXACT4 with matrix labels
    labs = [l for l in EXACT4 if l in ov_df.index and l in ov_df.columns]
    if labs:
        sub = ov_df.loc[labs, labs].copy()
        M = sub.to_numpy(float)
        fig = plt.figure(figsize=(5.8, 5.1))
        im = plt.imshow(np.clip(M, 0, 1), vmin=0, vmax=1, aspect="equal")
        plt.xticks(np.arange(len(labs)), labs, rotation=30, ha="right")
        plt.yticks(np.arange(len(labs)), labs)
        plt.colorbar(im, fraction=0.046, pad=0.04, label="overlap")
        plt.title("Signature overlap (four-lineage subset)")
        fig.tight_layout()
        _savefig(fig, "overlap_four_lineages")
        plt.show(); plt.close(fig)
    else:
        print("Overlap matrix does not contain the chosen four lineages.")
else:
    print("overlap_matrix.csv not available; skipping overlap heatmap.")


In [ ]:

# ---- Coverage distribution & simplex QC ----
if residuals_df is not None and not residuals_df.empty:
    fig = plt.figure(figsize=(7.5, 4.6))
    cov = residuals_df["coverage"].dropna().to_numpy(float)
    plt.hist(np.clip(cov, 0, np.nanpercentile(cov, 99)), bins=80, alpha=0.85)
    plt.title("Coverage distribution (clipped at 99th percentile)")
    plt.xlabel("coverage"); plt.ylabel("count")
    fig.tight_layout()
    _savefig(fig, "coverage_hist")
    plt.show(); plt.close(fig)

# simplex satisfied table
simp_df = None
simp_path = Path(ALT_TABLES_DIR) / "simplex_satisfied.csv"
if simp_path.exists():
    simp_df = pd.read_csv(simp_path)
    print("Simplex satisfied table:")
    display(simp_df)

# Small summary counts
n_samples = theta_df[["site_id","date","sample_id"]].drop_duplicates().shape[0]
n_sites = theta_df["site_id"].nunique()
print(f"Samples: {n_samples}, Sites: {n_sites}")



## Done

- Global and per-site lineage shares (four lineages + optional "Other")
- Residual diagnostics (if residuals available)
- Growth (d/dt logit share), doubling times of odds, peak timing
- Cross-lineage correlation of daily Δshare
- Uncertainty bands (if theta_uncertainty available)
- Objective traces (if objective_trace available)
- Overlap matrix (four-lineage subset if available)
- Coverage distribution and simplex check

Set `SAVE_FIGS=True` to export PNGs and CSVs under `OUTPUT_DIR`.
